### WHY DID WE CREATED A FILE DISCOVERY FRAME WORK? 

> Framework scans the adls gen2 landing zone and compares incoming files against the metadata configuration table. 

In [0]:
from pyspark.sql.functions import col, regexp_extract, current_timestamp, lit

source_name = "netflix"
landing_path = "/Volumes/bronze/metadata/landing_files/*/*.csv"

files_df = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(landing_path)
    .select(
        col("path").alias("discovered_file_path"),
        regexp_extract(col("path"), r"([^/]+\.csv)$", 1).alias("discovered_file_name"),
        regexp_extract(col("path"), r"ingest_dt=([^/]+)", 1).alias("ingest_dt")
    )
    .withColumn("source_name", lit(source_name))
)

config_files_df = spark.sql("""
SELECT DISTINCT file_pattern
FROM bronze.metadata.bronze_config
WHERE is_active = true
""").select(
    regexp_extract(col("file_pattern"), r"([^/]+\.csv)$", 1).alias("configured_file_name")
)

unknown_files_df = (
    files_df.alias("f")
    .join(
        config_files_df.alias("c"),
        col("f.discovered_file_name") == col("c.configured_file_name"),
        "left_anti"
    )
    .withColumn("discovery_status", lit("NEW_FILE_NOT_CONFIGURED"))
    .withColumn("discovered_timestamp", current_timestamp())
    .withColumn("comments", lit("File exists in landing but not available in bronze_config"))
    .select(
        "source_name",
        "discovered_file_path",
        "discovered_file_name",
        "ingest_dt",
        "discovery_status",
        "discovered_timestamp",
        "comments"
    )
)

unknown_files_df.write.mode("append").saveAsTable(
    "bronze.metadata.bronze_file_discovery"
)